In [3]:
from transformers import AutoProcessor

txt = "<thinking> with Maria.rossi1993@gmail.com. The email address has been entered correctly, but I need to ensure that it is confirmed and the email content is shared. I should look for a way to confirm the selection or proceed with sharing the document.\n</thinking>\n<tool_call>\n{\"name\": \"mobile_use\", \"arguments\": {\"action\": \"click\", \"coordinate\": [857, 203]}}\n</tool_call>"

processor = AutoProcessor.from_pretrained("/root/workspace/datasets/share/Qwen2.5-VL-3B-Instruct")

inputs = processor(text=txt, return_tensors="pt", padding=True, truncation=True)

print(inputs)

{'input_ids': tensor([[ 13708,  15736,     29,    448,  23016,     13,   2128,     72,     16,
             24,     24,     18,  10375,    905,     13,    576,   2551,   2621,
            702,   1012,  10636,  12440,     11,    714,    358,   1184,    311,
           5978,    429,    432,    374,  10774,    323,    279,   2551,   2213,
            374,   6094,     13,    358,   1265,   1401,    369,    264,   1616,
            311,   7683,    279,   6589,    476,  10354,    448,  11560,    279,
           2197,    624,    522,  82260,    397, 151657,    198,   4913,    606,
            788,    330,  14933,  15951,    497,    330,  16370,    788,   5212,
           1311,    788,    330,   3678,    497,    330,  62526,    788,    508,
             23,     20,     22,     11,    220,     17,     15,     18,     60,
          11248, 151658]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import torch

model_path = '/root/workspace/datasets/share/share1/Qwen2.5-VL-3B-Instruct'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="cuda",
)
processor = AutoProcessor.from_pretrained(model_path)

nonthink_question_template = (
        f"In this UI screenshot, I want to perform the command '{task_prompt}'.\n"
        "Please provide the action to perform (enumerate in ['click', 'scroll']) and the coordinate where the cursor is moved to(integer) if click is performed.\n"
        "Directly output final answer in <answer> </answer> tags."
        "The output answer format should be as follows:\n"
        "<answer>[{'action': enum['click', 'scroll'], 'coordinate': [x, y]}]</answer>\n"
        "Please strictly follow the format."
    ) ## w/o think format

query = '<image>\n' + question_template
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path}
        ] + [{"type": "text", "text": query}],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)
# inputs = inputs.cuda()

generated_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
response = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
response = response[0]

print(response)